## 已接受历史树模型摘要回放

读取v2摘要和日期基数，并从历史值新画一张展示图；无模型拟合、重训或39逐行修复。真实研究blocked，另有fresh CLI但此处不调用。

[本语言逐本运行模式与准备条件](../../../NOTEBOOK_GUIDE.md)。本阶段只制作语言层，便携环境与执行在后续隔离冷启动核验。

# 项目6 v2：树模型、稳定性与消融

历史v2继承v1标签、折、内层验证、组合和账本，新增树/RF/HGB及PCA+Ridge消融。HGB关闭内置早停，以内层日期折选轮数。冻结前已看v1构造结果，不是真实市场纯事前注册。

信号 $D_k$ 收盘、成交 $D_{k+1}$、标签成熟 $D_{k+6}$。教学源已接受，真实研究与正式最终验证仍未完成。

In [ ]:
from pathlib import Path
import os, sys, uuid
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir())
for path in (ROOT / "src", ROOT / "code", ROOT):
    if str(path) not in sys.path: sys.path.insert(0, str(path))
os.environ["MPLBACKEND"] = "Agg"
os.chdir(ROOT)
RUN_DIR = ROOT / "runs" / ("p06_trees_v2_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Run directory:", RUN_DIR.relative_to(ROOT))
import json, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
P06 = ROOT / "fixtures/projects/project06"
print("P6 trees: historical accepted CONSTRUCTED result display; no fitting/retraining; real route blocked")


## 1. 缩小拟合格已跳过

原八资产420日训练格现在只打印SKIPPED；旧“实时运行约20秒”不适用，不产生fresh训练结果。

In [ ]:
print("Original small model fitting cell SKIPPED in I2: no P6 retraining and no historical Ch39 repair")


## 2. 历史摘要与日期基数

读共享fixtures的v2_ic_date_bases及base/regime摘要。IC按自身有效日，delta IC按与Ridge交集。稀疏模型存在常数预测日；压缩NW(4)不是交易日HAC，故相应t显示NaN。baseline_check沿历史A8-A c3身份。仅从旧值新画historical_constructed_trees_sharpe.png，不是重训结果。

In [ ]:
# IC dates: own IC dates differ for sparse models (constant predictions on some dates). t-values are shown only for
# series with IC on every mature-label date; for the others the frozen summary's t was computed on a compressed
# series of valid dates and is not a trading-day NW(4) statistic (rework R01).
bases = json.loads((P06 / "v2_ic_date_bases.json").read_text())["scenarios"]
rows = []
for scen in ("base", "regime"):
    s = json.loads((P06 / f"v2_summary_{scen}.json").read_text())
    assert s["data_class"] == "CONSTRUCTED" and s["baseline_check"]["ok"]
    for m, r in s["metrics"].items():
        if m in ("zero", "train_mean"):
            continue
        d, b = r.get("delta_ic_vs_ridge", {}), bases[scen][m]
        assert b["own_mean_matches_frozen"] and b.get("delta_matches_frozen", True)
        contiguous = b["frozen_t_is_trading_day_hac"]
        rows.append({"scenario": scen, "model": m, "R2_vs_mean_%": 100 * r["prediction"]["oos_r2_vs_train_mean"],
                     "IC": r["prediction"]["ic_mean"], "IC_dates": b["own_ic_dates"],
                     "IC_t": r["prediction"]["ic_t"] if contiguous else np.nan,
                     "dIC_vs_ridge": d.get("mean"), "paired_dates": b.get("paired_dates"),
                     "ridge_on_paired": b.get("ridge_paired_mean"), "dIC_t": d.get("t_nw") if contiguous else np.nan,
                     "net10_SR": r["portfolio"]["0.001"]["sharpe"], "net20_SR": r["portfolio"]["0.002"]["sharpe"],
                     "turnover": r["portfolio"]["0.001"]["turnover_ann"]})
full = pd.DataFrame(rows).set_index(["scenario", "model"])
display(full.round(4))
print("NaN t = not reported: the series has dates without a defined IC (see v2_ic_date_bases.json).")
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
for a, scen in zip(ax, ("base", "regime")):
    t = full.loc[scen]
    a.bar(t.index, t["net10_SR"], color=["#247F82" if m in ("tree", "rf", "gbm") else "#7F7F7F" for m in t.index])
    a.axhline(0, color="#444444", lw=0.6); a.set_title(f"{scen}: Sharpe after 10 bp (CONSTRUCTED)"); a.tick_params(axis="x", rotation=60)
plt.tight_layout(); fig.savefig(RUN_DIR / "historical_constructed_trees_sharpe.png",dpi=130); plt.show()

## 3. 消融与重要性稳定性

In [ ]:
s = json.loads((P06 / "v2_summary_base.json").read_text())
abl = pd.DataFrame({k: {"IC": v["prediction"]["ic_mean"], "dIC_vs_full": v["delta_ic_vs_full"]["mean"],
                        "net10_SR": v["portfolio"]["0.001"]["sharpe"]} for k, v in s["ablation"].items()}).T
display(abl.round(4))
print({m: round(v["consecutive_rank_corr_mean"], 3) for m, v in s["importance_stability"].items()})
print({m: {sd: round(r["portfolio"]["0.001"]["sharpe"], 3) for sd, r in v.items() if sd != "prediction_corr"} for m, v in s["seeds"].items()})